<a href="https://colab.research.google.com/github/Priyanshu27083/DL_practice/blob/main/CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import zipfile
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
gtsrb_download_url = 'https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/GTSRB_Final_Training_Images.zip'

# Path where the zip file will be saved
gtsrb_zip_path = '/content/GTSRB_Final_Training_Images.zip'
output_dir = '/content/' # Extract directly into /content to simplify subsequent paths

# Create output directory if it doesn't exist (already created by colab usually)
os.makedirs(output_dir, exist_ok=True)

# Download the GTSRB dataset if it doesn't exist
if not os.path.exists(gtsrb_zip_path):
    print(f"Downloading GTSRB dataset from {gtsrb_download_url}...")
    !wget -O {gtsrb_zip_path} {gtsrb_download_url}
    print("Download complete.")
else:
    print("GTSRB zip file already exists. Skipping download.")

print(f"Unzipping GTSRB dataset to: {output_dir}")
with zipfile.ZipFile(gtsrb_zip_path, 'r') as zip_ref:
    zip_ref.extractall(output_dir)

print("GTSRB dataset unzipped.")

--2026-04-07 04:50:42--  https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/GTSRB_Final_Training_Images.zip
Resolving sid.erda.dk (sid.erda.dk)... 130.225.104.13
Connecting to sid.erda.dk (sid.erda.dk)|130.225.104.13|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 276294756 (263M) [application/zip]
Saving to: ‘/content/GTSRB_Final_Training_Images.zip’

/content/GTSRB_Fina 100%[===================>] 263.50M  14.7MB/s    in 18s     

2026-04-07 04:51:00 (14.9 MB/s) - ‘/content/GTSRB_Final_Training_Images.zip’ saved [276294756/276294756]

Download complete.
Unzipping GTSRB dataset to: /content/
GTSRB dataset unzipped.


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Define the CNN model architecture
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

# Compile the model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Display model summary
model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 28, 28, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 12, 12, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 6, 6, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 4, 4, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 2, 2, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 43)             │         5,547 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 164,459 (642.42 KB)

 Trainable params: 164,459 (642.42 KB)

 Non-trainable params: 0 (0.00 B)

## Preprocessing the GTSRB Dataset

In [ ]:
import os
import pandas as pd
from PIL import Image
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

# Define the base directory where the dataset is extracted
base_dir = '/content/GTSRB/Final_Training/Images/'

# List to store image paths and their corresponding labels
images = []
labels = []

# Loop through each class folder (0-42)
for i in range(43):
    path = os.path.join(base_dir, str(i).zfill(5))
    # Read the CSV file for the current class to get image names and labels
    csv_file = pd.read_csv(os.path.join(path, 'GT-' + str(i).zfill(5) + '.csv'), sep=';')

    # Iterate through each row in the CSV
    for row_idx, row in csv_file.iterrows():
        img_path = os.path.join(path, row['Filename'])
        images.append(img_path)
        labels.append(row['ClassId'])

# Convert to numpy arrays
images = np.array(images)
labels = np.array(labels)

print(f"Total images found: {len(images)}")
print(f"Total labels found: {len(labels)}")

Total images found: 39209
Total labels found: 39209


### 2. Image Preprocessing (Resizing and Normalization)

In [ ]:
IMG_HEIGHT = 30
IMG_WIDTH = 30

def preprocess_image(image_path, target_height=IMG_HEIGHT, target_width=IMG_WIDTH):
    try:
        img = Image.open(image_path)
        img = img.resize((target_width, target_height)) # Resize image
        img = np.array(img) # Convert to numpy array
        return img
    except Exception as e:
        print(f"Error loading image {image_path}: {e}")
        return None

# Process all images
processed_images = []
for i, img_path in enumerate(images):
    img = preprocess_image(img_path)
    if img is not None:
        processed_images.append(img)
    else:
        # If an image fails to load, remove its corresponding label
        labels = np.delete(labels, i)

processed_images = np.array(processed_images)

# Normalize pixel values to be between 0 and 1
processed_images = processed_images / 255.0

print(f"Shape of processed images: {processed_images.shape}")
print(f"Shape of labels after potential removal: {labels.shape}")

Shape of processed images: (39209, 30, 30, 3)
Shape of labels after potential removal: (39209,)


### 3. Splitting Data and One-Hot Encoding Labels

In [ ]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    processed_images, labels, test_size=0.2, random_state=42, stratify=labels
)

# Convert labels to one-hot encoding
num_classes = len(np.unique(labels)) # Assuming labels range from 0 to num_classes-1
y_train_one_hot = to_categorical(y_train, num_classes=num_classes)
y_test_one_hot = to_categorical(y_test, num_classes=num_classes)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train_one_hot shape: {y_train_one_hot.shape}")
print(f"y_test_one_hot shape: {y_test_one_hot.shape}")

X_train shape: (31367, 30, 30, 3)
X_test shape: (7842, 30, 30, 3)
y_train_one_hot shape: (31367, 43)
y_test_one_hot shape: (7842, 43)


## 4. Building and Training the CNN Model

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Define the CNN model architecture based on the user's request, adapted for GTSRB dataset
model = Sequential([
    Conv2D(25, kernel_size=(3,3), strides=(1,1), padding='valid', activation='relu', input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    MaxPooling2D(pool_size=(4,4)), # Changed from (1,1) to (2,2) for effective pooling
    Flatten(),
    Dense(100, activation='relu'),
    Dense(num_classes, activation='softmax') # Using num_classes for GTSRB output
])

# Compile the model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Display model summary
model.summary()

NameError: name 'IMG_HEIGHT' is not defined

### Training the Model

In [ ]:
epochs = 10
history = model.fit(
    X_train, y_train_one_hot,
    epochs=epochs,
    validation_data=(X_test, y_test_one_hot)
)

print("Model training complete.")

Epoch 1/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 52s 50ms/step - accuracy: 0.4585 - loss: 1.8980 - val_accuracy: 0.8717 - val_loss: 0.5163
Epoch 2/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 79s 47ms/step - accuracy: 0.8573 - loss: 0.4614 - val_accuracy: 0.9611 - val_loss: 0.1471
Epoch 3/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 46s 47ms/step - accuracy: 0.9311 - loss: 0.2267 - val_accuracy: 0.9709 - val_loss: 0.0970
Epoch 4/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 81s 46ms/step - accuracy: 0.9543 - loss: 0.1496 - val_accuracy: 0.9856 - val_loss: 0.0589
Epoch 5/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 83s 48ms/step - accuracy: 0.9662 - loss: 0.1083 - val_accuracy: 0.9880 - val_loss: 0.0532
Epoch 6/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 46s 46ms/step - accuracy: 0.9731 - loss: 0.0856 - val_accuracy: 0.9883 - val_loss: 0.0408
Epoch 7/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 49s 49ms/step - accuracy: 0.9801 - loss: 0.0679 - val_accuracy: 0.9899 - val_loss: 0.0357
Epoch 8/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 46s 46ms/step - accuracy: 0.9807 - loss: 0.0624 - 

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Build CNN model
model = Sequential()

# 1st Conv Layer
model.add(Conv2D(32, (3,3), activation='relu', input_shape=(30,30,3)))
model.add(MaxPooling2D(pool_size=(2,2)))

# 2nd Conv Layer
model.add(Conv2D(64, (3,3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))

# 3rd Conv Layer
model.add(Conv2D(128, (3,3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))

# Flatten
model.add(Flatten())

# Fully Connected Layer
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.5))   # prevent overfitting

# Output Layer (43 classes)
model.add(Dense(43, activation='softmax'))

# Compile
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train
history = model.fit(
    X_train, y_train_one_hot,
    batch_size=64,
    epochs=15,
    validation_data=(X_test, y_test_one_hot)
)

Epoch 1/15
491/491 ━━━━━━━━━━━━━━━━━━━━ 48s 94ms/step - accuracy: 0.4269 - loss: 2.0069 - val_accuracy: 0.8509 - val_loss: 0.5610
Epoch 2/15
491/491 ━━━━━━━━━━━━━━━━━━━━ 79s 88ms/step - accuracy: 0.8812 - loss: 0.3895 - val_accuracy: 0.9630 - val_loss: 0.1442
Epoch 3/15
491/491 ━━━━━━━━━━━━━━━━━━━━ 81s 87ms/step - accuracy: 0.9492 - loss: 0.1726 - val_accuracy: 0.9802 - val_loss: 0.0738
Epoch 4/15
491/491 ━━━━━━━━━━━━━━━━━━━━ 43s 89ms/step - accuracy: 0.9691 - loss: 0.1056 - val_accuracy: 0.9870 - val_loss: 0.0547
Epoch 5/15
491/491 ━━━━━━━━━━━━━━━━━━━━ 44s 90ms/step - accuracy: 0.9784 - loss: 0.0720 - val_accuracy: 0.9890 - val_loss: 0.0429
Epoch 6/15
491/491 ━━━━━━━━━━━━━━━━━━━━ 80s 87ms/step - accuracy: 0.9850 - loss: 0.0526 - val_accuracy: 0.9901 - val_loss: 0.0387
Epoch 7/15
491/491 ━━━━━━━━━━━━━━━━━━━━ 43s 87ms/step - accuracy: 0.9860 - loss: 0.0451 - val_accuracy: 0.9917 - val_loss: 0.0350
Epoch 8/15
491/491 ━━━━━━━━━━━━━━━━━━━━ 44s 90ms/step - accuracy: 0.9884 - loss: 0.0389 - 